# ch05_facilities_value_chain — 5.10 NeqSim — a simplified topside

From chapter `ch05_facilities_value_chain`, section: **5.10 NeqSim — a simplified topside**.

This companion notebook reproduces or expands the textbook code block. Run all cells from top to bottom; cells that are intentionally illustrative are marked in the code comments.



In [ ]:
from neqsim import jneqsim as ns

# 1. Reservoir feed (~ 80 bar, 80 °C, North Sea oil)
fluid = ns.thermo.system.SystemSrkEos(353.15, 80.0)
for c, x in [("nitrogen", 0.005), ("CO2", 0.018),
             ("methane", 0.55), ("ethane", 0.07),
             ("propane", 0.05),  ("n-butane", 0.04),
             ("n-pentane", 0.03), ("n-hexane", 0.02),
             ("water", 0.05)]:
    fluid.addComponent(c, x)
fluid.addPlusFraction("C7", 0.21, 205.0 / 1000.0, 0.832)
fluid.getCharacterization().getLumpingModel().setNumberOfLumpedComponents(5)
fluid.getCharacterization().characterisePlusFraction()
fluid.setMixingRule("classic")
fluid.createDatabase(True)

# 2. Process system: feed -> HP sep -> MP sep -> LP sep
ProcessSystem = ns.process.processmodel.ProcessSystem
Stream        = ns.process.equipment.stream.Stream
Sep           = ns.process.equipment.separator.ThreePhaseSeparator
Valve         = ns.process.equipment.valve.ThrottlingValve

feed = Stream("wellstream", fluid)
feed.setFlowRate(8000.0, "kg/hr")
feed.setTemperature(80.0, "C"); feed.setPressure(80.0, "bara")

hp = Sep("HP sep",  feed)
v1 = Valve("v1", hp.getOilOutStream()); v1.setOutletPressure(20.0, "bara")
mp = Sep("MP sep",  v1.getOutletStream())
v2 = Valve("v2", mp.getOilOutStream()); v2.setOutletPressure(2.0, "bara")
lp = Sep("LP sep",  v2.getOutletStream())

p = ProcessSystem()
for u in (feed, hp, v1, mp, v2, lp): p.add(u)
p.run()

print("HP gas (kg/hr):", hp.getGasOutStream().getFlowRate("kg/hr"))
print("LP oil (kg/hr):", lp.getOilOutStream().getFlowRate("kg/hr"))


HP gas (kg/hr): 2421.8419042130986
LP oil (kg/hr): 4387.523313158613
